# Analyse de base de données environnementale et de confort thermique prenant en compte la vitesse d’air et les conditions chaude et humide.
## ETAPE 0

In [1]:
import pandas as pd

URL_META = "https://github.com/CenterForTheBuiltEnvironment/ashrae-db-II/raw/master/v2.1.0/db_metadata.csv"
URL_MEAS = "https://github.com/CenterForTheBuiltEnvironment/ashrae-db-II/raw/master/v2.1.0/db_measurements_v2.1.0.csv.gz"

meta = pd.read_csv(URL_META)
meas = pd.read_csv(URL_MEAS)
# on regarde le contenu des deux bases
print("metadata :", meta.shape)
print("measurements :", meas.shape)

df = meas.merge(
    meta,
    on="building_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_meta")
)
print("base fusionnée :", df.shape)

metadata : (809, 23)
measurements : (109033, 53)
base fusionnée : (109033, 75)


C:\Users\yanis\AppData\Local\Temp\ipykernel_3644\1423825777.py:7: DtypeWarning: Columns (5,35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  meas = pd.read_csv(URL_MEAS)


In [2]:
print(df.shape)
print('---------------')
print(df.info)
print('---------------')
print(df.columns)
print('---------------')
df.to_csv('maBaseDeTravailDuPFE.csv', index=False)

(109033, 75)
---------------
<bound method DataFrame.info of          index  record_id  building_id             timestamp   season  \
0            0          1            1  1995-05-18T00:00:00Z   winter   
1            1          2            1  1995-05-18T00:00:00Z   winter   
2            2          3            1  1995-05-18T00:00:00Z   winter   
3            3          4            1  1995-05-18T00:00:00Z   winter   
4            4          5            1  1995-05-18T00:00:00Z   winter   
...        ...        ...          ...                   ...      ...   
109028  110065     110066          809  2014-06-02T00:00:00Z  hot/wet   
109029  110066     110067          809  2014-06-02T00:00:00Z  hot/wet   
109030  110067     110068          809  2014-06-02T00:00:00Z  hot/wet   
109031  110068     110069          809  2014-06-02T00:00:00Z  hot/wet   
109032  110069     110070          809  2014-06-02T00:00:00Z  hot/wet   

       subject_id  age  gender  ht  wt  ...  records  has_age 

In [3]:
for col in df.select_dtypes(include=["object", "string"]):
    print(f"{col} ({df[col].nunique()} valeurs) :")
    print(df[col].unique())
    print()

timestamp (2836 valeurs) :
['1995-05-18T00:00:00Z' '1995-05-19T00:00:00Z' '1995-05-22T00:00:00Z' ...
 '2014-05-04T00:00:00Z' '2014-06-01T00:00:00Z' '2014-06-02T00:00:00Z']

season (6 valeurs) :
['winter' 'summer' 'cool/dry' 'hot/wet' 'spring' 'autumn' nan]

subject_id (2651 valeurs) :
[10.0 14.0 15.0 ... '703' '701' '815']

gender (3 valeurs) :
['female' 'male' nan 'undefined']

thermal_acceptability (2 valeurs) :
['acceptable' 'unacceptable' nan]

thermal_preference (3 valeurs) :
['cooler' 'no change' 'warmer' nan]

air_movement_acceptability (2 valeurs) :
['unacceptable' 'acceptable' nan]

air_movement_preference (3 valeurs) :
['more' 'no change' nan 'less']

building_id_inf (2 valeurs) :
['no' 'yes']

contributor (55 valeurs) :
['David Rowe' 'Fred Baumann' 'Gail Brager' 'Jill Brown' 'Richard de Dear'
 'Giovanna Donnini' 'John Busch' 'Nick Baker' 'Tri Karyono' 'Alison Kwok'
 'Jeetika Malik' 'Guy Newsham' 'Fergus Nicol' 'Nigel Oseland'
 'Salvatore Carlucci' 'Elisabetta Maria Patane' '

## Etape 1 exploration de la base
## Question 1 :

In [4]:
# A partir de "parameter_description" et de la liste des climats dispo (31)
CLIMATE_MAP = {
    "tropical wet savanna": ("Aw", "A", True),
    "tropical savanna": ("Aw", "A", True),
    "tropical monsoon": ("Am", "A", True),
    "tropical": ("A", "A", True),
    "wet equatorial": ("Af", "A", True),
    "tropical rainforest": ("Af", "A", True),
    "tropical dry savanna": ("As", "A", True),
}

df["koppen_code"] = df["climate"].map(
    lambda x: CLIMATE_MAP.get(x, (None, None, None))[0]
)

df["koppen_group"] = df["climate"].map(
    lambda x: CLIMATE_MAP.get(x, (None, None, None))[1]
)

df["is_tropical"] = df["climate"].map(
    lambda x: CLIMATE_MAP.get(x, (None, None, None))[2]
)

trop = df[df["is_tropical"] == True].copy()

print("Nombre total d'observations :", len(df))
print("Nombre d'observations tropicales :", len(trop))
print("Nombre de bâtiments tropicaux :", trop["building_id"].nunique())

Nombre total d'observations : 109033
Nombre d'observations tropicales : 21974
Nombre de bâtiments tropicaux : 253


In [5]:
print("Répartition des climats tropicaux :")
print(trop["climate"].value_counts())

print("\nRépartition par type de refroidissement :")
print(trop["cooling_type"].value_counts())

print("\nRépartition par type de bâtiment :")
print(trop["building_type"].value_counts())

Répartition des climats tropicaux :
climate
tropical wet savanna    8383
tropical savanna        6980
tropical monsoon        2101
tropical                1594
wet equatorial          1414
tropical rainforest     1350
tropical dry savanna     152
Name: count, dtype: int64

Répartition par type de refroidissement :
cooling_type
naturally ventilated    8875
air conditioned         8654
mixed mode              4445
Name: count, dtype: int64

Répartition par type de bâtiment :
building_type
office                 15468
multifamily housing     4161
classroom               2245
others                   100
Name: count, dtype: int64


In [6]:
# Nombre total d'observations tropicales
n_total = len(trop)

# Palier 1 : Tdb + RH
P1 = trop["ta"].notna() & trop["rh"].notna()

# Palier 2 : Tdb + RH + Tg/Tr
P2 = P1 & (trop["tg"].notna() | trop["tr"].notna())

# Palier 3 : Tdb + RH + Tg/Tr + V
P3 = P2 & trop["vel"].notna()

# Palier 4 : Tdb + RH + Tg/Tr + V + clo
P4 = P3 & trop["clo"].notna()

# Palier 5 : Tdb + RH + Tg/Tr + V + clo + métabolisme et/ou activité/fonction du bâtiment
has_met = trop["met"].notna()

has_activity = trop[
    ["activity_10", "activity_20", "activity_30", "activity_60"]
].notna().any(axis=1)

has_building = trop["building_type"].notna()

P5 = P4 & (has_met | has_activity | has_building)

# Résultats
resultats_q1 = pd.DataFrame({
    "Palier": [
        "P1 : Tdb + RH",
        "P2 : Tdb + RH + Tg/Tr",
        "P3 : Tdb + RH + Tg/Tr + V",
        "P4 : Tdb + RH + Tg/Tr + V + clo",
        "P5 : Tdb + RH + Tg/Tr + V + clo + métabolisme et/ou activité/fonction du bâtiment"
    ],
    "Nombre d'observations": [
        P1.sum(),
        P2.sum(),
        P3.sum(),
        P4.sum(),
        P5.sum()
    ]
})

resultats_q1["Pourcentage"] = (
    resultats_q1["Nombre d'observations"] / n_total * 100
).round(1)

resultats_q1

,Palier,Nombre d'observations,Pourcentage
0,P1 : Tdb + RH,21914,99.7
1,P2 : Tdb + RH + Tg/Tr,14104,64.2
2,P3 : Tdb + RH + Tg/Tr + V,9964,45.3
3,P4 : Tdb + RH + Tg/Tr + V + clo,9730,44.3
4,P5 : Tdb + RH + Tg/Tr + V + clo + métabolisme ...,9730,44.3


In [7]:
print("Observations P4 :", P4.sum())
print("P4 + met :", (P4 & has_met).sum())
print("P4 + activité :", (P4 & has_activity).sum())
print("P4 + fonction/type bâtiment :", (P4 & has_building).sum())
print("P4 + met OU activité OU bâtiment :", P5.sum())

Observations P4 : 9730
P4 + met : 9730
P4 + activité : 3421
P4 + fonction/type bâtiment : 9730
P4 + met OU activité OU bâtiment : 9730


In [8]:
# Attribution du palier maximal atteint pour chaque observation

trop["tier"] = 0

trop.loc[P1, "tier"] = 1
trop.loc[P2, "tier"] = 2
trop.loc[P3, "tier"] = 3
trop.loc[P4, "tier"] = 4
trop.loc[P5, "tier"] = 5

In [9]:
# Une ligne par building_id tropical

study = (
    trop.groupby("building_id")
    .agg(
        publication=("publication", "first"),
        country=("country", "first"),
        city=("city", "first"),
        climate_label=("climate", "first"),
        koppen=("koppen_code", "first"),
        building_type=("building_type", "first"),
        cooling_type=("cooling_type", "first"),
        n_votes=("record_id", "count"),
        max_tier=("tier", "max"),
    )
    .reset_index()
    .sort_values("n_votes", ascending=False)
)

study

,building_id,publication,country,city,climate_label,koppen,building_type,cooling_type,n_votes,max_tier
224,654,https://doi.org/10.1016/j.buildenv.2014.01.002,india,chennai,tropical wet savanna,Aw,office,mixed mode,1527,1
214,592,https://doi.org/10.1016/j.buildenv.2009.06.005,brazil,maceio,tropical monsoon,Am,classroom,naturally ventilated,1160,5
226,656,https://doi.org/10.1016/j.buildenv.2014.01.002,india,chennai,tropical wet savanna,Aw,office,mixed mode,1127,1
213,591,https://doi.org/10.1016/j.buildenv.2009.06.005,brazil,maceio,tropical monsoon,Am,classroom,naturally ventilated,914,5
244,735,https://doi.org/10.1016/j.buildenv.2016.09.024,malaysia,kuala lumpur,tropical,A,office,air conditioned,629,5
...,...,...,...,...,...,...,...,...,...,...
94,324,https://doi.org/10.1016/j.buildenv.2022.109187,india,bengaluru,tropical wet savanna,Aw,multifamily housing,mixed mode,1,5
95,325,https://doi.org/10.1016/j.buildenv.2022.109187,india,bengaluru,tropical wet savanna,Aw,multifamily housing,naturally ventilated,1,5
96,326,https://doi.org/10.1016/j.buildenv.2022.109187,india,bengaluru,tropical wet savanna,Aw,multifamily housing,mixed mode,1,5
97,327,https://doi.org/10.1016/j.buildenv.2022.109187,india,bengaluru,tropical wet savanna,Aw,multifamily housing,naturally ventilated,1,5


In [10]:
study.to_csv("tropical_studies_list.csv", index=False)

In [11]:
# Regroupement par publication
pub = (
    trop.groupby("publication", dropna=False)
    .agg(
        n_buildings=("building_id", "nunique"),
        countries=("country", lambda x: sorted(set(x.dropna()))),
        koppen=("koppen_code", lambda x: sorted(set(x.dropna()))),
        building_types=("building_type", lambda x: sorted(set(x.dropna()))),
        cooling_types=("cooling_type", lambda x: sorted(set(x.dropna()))),
        n_votes=("record_id", "count"),
        max_tier=("tier", "max"),
        min_tier=("tier", "min"),
    )
    .reset_index()
    .sort_values("n_votes", ascending=False)
)

pub


,publication,n_buildings,countries,koppen,building_types,cooling_types,n_votes,max_tier,min_tier
2,"Kwok, A.G. (1998). Thermal Comfort in Tropical...",12,[usa],[Aw],[office],"[air conditioned, naturally ventilated]",3544,2,2
10,https://doi.org/10.1016/j.buildenv.2014.01.002,4,[india],[Aw],[office],"[air conditioned, mixed mode]",2907,1,1
8,https://doi.org/10.1016/j.buildenv.2009.06.005,2,[brazil],[Am],[classroom],[naturally ventilated],2074,5,5
11,https://doi.org/10.1016/j.buildenv.2015.12.019,8,[india],[Aw],[office],"[air conditioned, naturally ventilated]",2029,1,1
12,https://doi.org/10.1016/j.buildenv.2016.09.024,8,"[indonesia, malaysia, singapore]",[A],[office],"[air conditioned, mixed mode, naturally ventil...",1594,5,3
5,"de Dear, R.J. & Fountain, M.E. (1994). Field e...",23,[australia],[Aw],[office],[air conditioned],1231,5,5
7,https://doi.org/10.1016/0378-7788(92)90016-A,5,[thailand],[Aw],[office],"[air conditioned, naturally ventilated]",1160,5,3
13,https://doi.org/10.1016/j.buildenv.2022.109187,140,[india],[Aw],[multifamily housing],"[mixed mode, naturally ventilated]",1070,5,5
15,https://doi.org/10.1016/j.renene.2012.01.017,5,[mexico],"[As, Aw]",[multifamily housing],[mixed mode],1055,1,1
4,"de Dear, R. J. & Auliciems, A. (1985). Validat...",15,[australia],[Aw],[office],[air conditioned],1045,5,5


In [12]:
pub.to_csv("tropical_publications_list.csv", index=False)

In [13]:
# Verification : est-ce que toutes les études tropicales ont une publication renseignée ?
n_missing_pub = trop["publication"].isna().sum()
print("Votes tropicaux sans publication renseignée :", n_missing_pub)

missing_pub_buildings = (
    trop[trop["publication"].isna()]
    .groupby("building_id")
    .agg(
        country=("country", "first"),
        city=("city", "first"),
        climate=("climate", "first"),
        building_type=("building_type", "first"),
        cooling_type=("cooling_type", "first"),
        contributor=("contributor", "first"),
        n_votes=("record_id", "count"),
    )
)
missing_pub_buildings

Votes tropicaux sans publication renseignée : 100


,country,city,climate,building_type,cooling_type,contributor,n_votes
building_id,,,,,,,
808,malaysia,kota kinabalu,tropical rainforest,others,air conditioned,Harimi Djamila,12
809,malaysia,kota kinabalu,tropical rainforest,others,air conditioned,Harimi Djamila,88


## Question 2 :

In [14]:
# Defini tout ce qui n'est pas dans "is_tropical"
non_trop = df[df["is_tropical"] != True].copy()
climats_hors_tropical = (
    non_trop["climate"]
    .value_counts()
    .rename_axis("climate")
    .reset_index(name="n_observations")
)

climats_hors_tropical

,climate,n_observations
0,hot-summer mediterranean,21722
1,humid subtropical,15257
2,hot semi-arid,8642
3,warm-summer mediterranean,5773
4,mediterranean,4584
5,temperate oceanic,4422
6,monsoon-influenced hot-summer humid continental,3739
7,warm-summer humid continental,2990
8,hot desert,2083
9,desert (hot arid),2011


In [15]:
print("Votes hors tropical :", len(non_trop))
print("Bâtiments hors tropical :", non_trop["building_id"].nunique())

Votes hors tropical : 87059
Bâtiments hors tropical : 556


In [16]:
non_trop[["ta","rh","vel","clo"]].describe(percentiles=[.25,.5,.75]).round(2)

,ta,rh,vel,clo
count,87059.00,80320.00,75911.00,82070.00
mean,24.18,45.41,0.16,0.69
std,3.66,15.60,0.25,0.29
min,10.10,2.00,0.00,0.03
25%,22.12,33.20,0.04,0.50
50%,23.60,44.35,0.08,0.65
75%,26.00,56.00,0.17,0.79
max,39.80,100.00,3.97,4.00


In [17]:
# Sous-ensemble de référence : palier 4 uniquement (Tdb+RH+Tg/Tr+V+clo complets)
trop_complete = trop[P4].copy()
print("n observations :", len(trop_complete))
print("n bâtiments :", trop_complete["building_id"].nunique())

n observations : 9730
n bâtiments : 199


In [18]:
print("Climat :\n", trop_complete["climate"].value_counts())
print("\nPays (n =", trop_complete["country"].nunique(), ") :\n", trop_complete["country"].value_counts())
print("\nType de bâtiment :\n", trop_complete["building_type"].value_counts())
print("\nMode de refroidissement :\n", trop_complete["cooling_type"].value_counts())

Climat :
 climate
tropical savanna        3422
tropical monsoon        2101
tropical wet savanna    1799
tropical                1591
wet equatorial           817
Name: count, dtype: int64

Pays (n = 7 ) :
 country
australia    2276
brazil       2125
india        1775
thailand     1146
malaysia     1135
singapore     873
indonesia     400
Name: count, dtype: int64

Type de bâtiment :
 building_type
office                 5881
classroom              2074
multifamily housing    1775
Name: count, dtype: int64

Mode de refroidissement :
 cooling_type
air conditioned         4576
naturally ventilated    4459
mixed mode               695
Name: count, dtype: int64


In [19]:
trop_complete[["ta","rh","vel","clo"]].describe(percentiles=[.25,.5,.75]).round(2)

,ta,rh,vel,clo
count,9730.00,9730.00,9730.00,9730.00
mean,26.10,61.20,0.29,0.49
std,3.04,10.47,0.36,0.18
min,18.30,23.00,0.00,0.05
25%,23.70,55.10,0.10,0.38
50%,25.70,61.00,0.16,0.48
75%,28.68,67.51,0.31,0.57
max,34.20,99.70,3.62,2.34


In [20]:
def paliers_completude(sub):
    has_tdb = sub["ta"].notna()
    has_rh  = sub["rh"].notna()
    has_tgtr = sub["tg"].notna() | sub["tr"].notna()
    has_v   = sub["vel"].notna()
    has_clo = sub["clo"].notna()
    has_met = sub["met"].notna()
    has_activity = sub[
    ["activity_10", "activity_20", "activity_30", "activity_60"]
    ].notna().any(axis=1)
    has_building = sub["building_type"].notna()


    P1 = has_tdb & has_rh
    P2 = P1 & has_tgtr
    P3 = P2 & has_v
    P4 = P3 & has_clo
    P5 = P4 & (has_met | has_activity | has_building)

    n = len(sub)
    return pd.DataFrame({
        "Palier": ["P1: Tdb+RH", "P2: +Tg/Tr", "P3: +V", "P4: +clo", "P5: + met et/ou act/fonc bât"],
        "n_votes": [P1.sum(), P2.sum(), P3.sum(), P4.sum(), P5.sum()],
        "%": [round(100*P1.sum()/n,1), round(100*P2.sum()/n,1),
              round(100*P3.sum()/n,1), round(100*P4.sum()/n,1), round(100*P5.sum()/n,1)],
    })

## Critère A 

In [21]:
climats_proches_label = [
    "humid subtropical",                    # Cfa - "humid" dans la définition Köppen
    "monsoon-influenced humid subtropical",  # Cwa - idem
    "hot semi-arid",                         # BSh - cas limite
]
crit_A = non_trop["climate"].isin(climats_proches_label)
print("Critère A - n votes :", crit_A.sum())
non_trop.loc[crit_A, "climate"].value_counts()

Critère A - n votes : 25685


climate
humid subtropical                       15257
hot semi-arid                            8642
monsoon-influenced humid subtropical     1786
Name: count, dtype: int64

In [22]:
res_A = paliers_completude(non_trop[crit_A])
res_A

,Palier,n_votes,%
0,P1: Tdb+RH,25197,98.1
1,P2: +Tg/Tr,10034,39.1
2,P3: +V,9697,37.8
3,P4: +clo,9689,37.7
4,P5: + met et/ou act/fonc bât,9689,37.7


## Critère B

In [23]:
# Seuils derives du sous-ensemble de reference
SEUIL_TDB = round(trop_complete["ta"].median(), 1)
SEUIL_RH  = round(trop_complete["rh"].median(), 1)
print("Seuils :", SEUIL_TDB, SEUIL_RH)

crit_B = (non_trop["ta"] >= SEUIL_TDB) & (non_trop["rh"] >= SEUIL_RH)
print("Critere B - n votes :", crit_B.sum())
non_trop.loc[crit_B, "climate"].value_counts()

Seuils : 25.7 61.0
Critere B - n votes : 5640


climate
humid subtropical                                  2448
hot semi-arid                                      1360
desert (hot arid)                                   511
hot desert                                          322
monsoon-influenced humid subtropical                316
temperate                                           258
monsoon-influenced hot-summer humid continental     172
hot-summer mediterranean                            107
semi arid midlatitude                                81
subtropical hot and dry                              19
temperate oceanic                                    19
semi arid high altitude                              15
mediterranean                                         5
warm-summer mediterranean                             5
monsoon-influenced temperate oceanic                  2
Name: count, dtype: int64

In [24]:
res_B = paliers_completude(non_trop[crit_B])
res_B

,Palier,n_votes,%
0,P1: Tdb+RH,5640,100.0
1,P2: +Tg/Tr,3295,58.4
2,P3: +V,3111,55.2
3,P4: +clo,3097,54.9
4,P5: + met et/ou act/fonc bât,3097,54.9


## Comparaison A et B

In [25]:
comparaison = pd.DataFrame({
    "Critère": ["A (label climatique)", "B (seuil mesuré)"],
    "n_votes": [int(crit_A.sum()), int(crit_B.sum())],
    "% P4 atteint": [res_A.loc[3, "%"], res_B.loc[3, "%"]],
})
print(comparaison)

seulement_B = crit_B & ~crit_A
print("\nVotes communs aux deux critères :", (crit_A & crit_B).sum())
print("Votes trouvés par B mais PAS par A :", seulement_B.sum())

                Critère  n_votes  % P4 atteint
0  A (label climatique)    25685          37.7
1      B (seuil mesuré)     5640          54.9

Votes communs aux deux critères : 4124
Votes trouvés par B mais PAS par A : 1516


In [26]:
cas_inattendus = non_trop[
    seulement_B & non_trop["climate"].isin(["hot desert", "desert (hot arid)"])
]
print("Votes désert qui remplissent quand même le seuil chaud-humide :", len(cas_inattendus))
cas_inattendus[["building_id","country","city","cooling_type","ta","rh"]].drop_duplicates("building_id")

Votes désert qui remplissent quand même le seuil chaud-humide : 833


,building_id,country,city,cooling_type,ta,rh
18130,172,pakistan,karachi,naturally ventilated,31.1,75.9
18559,173,pakistan,karachi,naturally ventilated,26.9,64.0
18790,174,pakistan,multan,naturally ventilated,29.6,77.0
20607,177,pakistan,quettar,naturally ventilated,39.4,62.3
51121,568,tunisia,gabes,naturally ventilated,26.3,62.0
51240,570,tunisia,gabes,naturally ventilated,30.8,63.0
107858,806,iran,bandar abbas,naturally ventilated,29.6,63.0


In [27]:
for bid in [172, 173, 174, 177, 568, 570, 806]:
    b = df[df["building_id"] == bid]
    print(f"building_id {bid} : n={len(b)}, RH médiane={b['rh'].median():.1f}, RH 75e percentile={b['rh'].quantile(.75):.1f}")

building_id 172 : n=190, RH médiane=66.5, RH 75e percentile=69.9
building_id 173 : n=470, RH médiane=43.7, RH 75e percentile=59.4
building_id 174 : n=436, RH médiane=69.0, RH 75e percentile=74.2
building_id 177 : n=491, RH médiane=16.7, RH 75e percentile=32.8
building_id 568 : n=70, RH médiane=60.0, RH 75e percentile=62.0
building_id 570 : n=72, RH médiane=57.5, RH 75e percentile=60.0
building_id 806 : n=587, RH médiane=61.0, RH 75e percentile=66.0


In [28]:
df[df["building_id"]==177][["ta","rh"]].sort_values("ta", ascending=False).head(10)

,ta,rh
20607,39.4,62.3
20517,38.4,53.7
20717,37.3,47.3
20609,36.3,57.9
20445,35.9,48.8
20608,35.9,58.3
20384,35.5,46.3
20542,35.2,10.9
20484,35.1,16.0
20387,34.7,52.5


In [29]:
meta[meta["building_id"] == 779]

,building_id,building_id_inf,contributor,publication,region,country,city,lat,lon,climate,...,records,has_age,has_ec,has_timestamp,timezone,met_source,isd_station,isd_distance,database,quality_assurance
778,779,yes,Shin-ichi Tanabe,https://doi.org/10.1080/00038628.2012.744296,asia,japan,tokyo,35.676192,139.650311,humid subtropical,...,118,no,no,no,Asia/Tokyo,original_data,NaN,NaN,2.0,pass


# YANIS

Violet : YANIS
13. Malik et Bardhan (2022) - Mumbai, Inde - logements - NV - 705 votes
14. Karyono (1996) - Jakarta, Indonésie - bureaux - AC+MM+NV - 596 voix - palier max 2
15. Akande et Adebamowo (2010) - Bauchi, Nigeria - logements - NV - 468 votes - palier max 1
16. Andamon (2006) - Makati, Philippines - bureaux - AC - 277 voix - palier max 1 (pas encore accès)
17. Sekhar et coll. (2003) - Singapour - bureaux - AC - 216 voix - palier max 3
18. Schweiker et coll. (2020) - Kota Kinabalu, Malaisie - salle de classe - AC - 171 votes - palier max 1
19. 2 bâtiments sans publication renseignée (Kota Kinabalu, Malaisie, contributrice Harimi Djamila) - 100 votes - palier max 1
20. Xavier (2000, thèse) - Brésil - bureaux - AC - 51 voix - palier max 4

# Texte 13 : 
Malik et Bardhan (2022) - Mumbai, Inde (Aw) - logements - 2 bâtiments - NV - 705 votes : Moonsoon 277 / Winter 253 / Summer 175 - Ventilation : Présente 
https://doi.org/10.1080/17512549.2021.1907224

In [30]:
masque = df["publication"].str.contains("17512549", na=False)
malik = df[masque].copy()
print("Pays :", malik["country"].value_counts())
print("Villes :", malik["city"].value_counts())
print("climat :", malik["climate"].value_counts())
print("saison :", malik["season"].value_counts())
print("Type de Bâtiments :", malik["building_type"].value_counts())
print("Bâtiments :", malik["building_id"].unique())
print("Genre :", malik["gender"].value_counts())
print("Types de ventilation :", malik["cooling_type"].value_counts())

print("Nombre de votes :", len(malik))



Pays : country
india    705
Name: count, dtype: int64
Villes : city
mumbai    705
Name: count, dtype: int64
climat : climate
tropical wet savanna    705
Name: count, dtype: int64
saison : season
hot/wet     452
cool/dry    253
Name: count, dtype: int64
Type de Bâtiments : building_type
multifamily housing    705
Name: count, dtype: int64
Bâtiments : [166 167]
Genre : gender
female    537
male      168
Name: count, dtype: int64
Types de ventilation : cooling_type
naturally ventilated    705
Name: count, dtype: int64
Nombre de votes : 705


In [31]:
variables = list(meas.columns)

remplissage = malik[variables].notna().mean() * 100
print(remplissage.round(1).sort_values().to_string())

heater                          0.0
activity_60                     0.0
activity_30                     0.0
activity_20                     0.0
activity_10                     0.0
ppd_ce                          0.0
vel_l                           0.0
vel_m                           0.0
vel_h                           0.0
tg_l                            0.0
tg_m                            0.0
thermal_comfort                 0.0
tr                              0.0
top                             0.0
air_movement_acceptability      0.0
ta_m                            0.0
rh_out                          0.0
t_out_monthly                   0.0
wt                              0.0
ht                              0.0
pmv_ce                          0.0
ppd                             0.0
ta_l                            0.0
subject_id                      0.0
set                             0.0
pmv                             0.0
blind_curtain                  51.8
thermal_sensation           

In [32]:
saisons_article = {8: "Monsoon", 9: "Monsoon", 1: "Winter", 5: "Summer"}

malik["date"] = pd.to_datetime(malik["timestamp"])
malik["mois"] = malik["date"].dt.month
malik["saison_article"] = malik["mois"].map(saisons_article)
print(malik["date"].value_counts().sort_index())
print(malik["mois"].value_counts().sort_index())

stats = malik.groupby("saison_article")[["t_out", "rh_out", "ta", "tg", "rh", "vel"]].agg(["mean", "min", "max"])
ordre = ["Monsoon", "Winter", "Summer"]
stats = stats.reindex(ordre)
print(stats.round(1).T)

date
2018-05-19 00:00:00+00:00    14
2018-08-19 00:00:00+00:00    45
2018-08-20 00:00:00+00:00    45
2018-09-02 00:00:00+00:00    42
2018-09-04 00:00:00+00:00    44
2018-09-05 00:00:00+00:00    49
2018-09-11 00:00:00+00:00    52
2019-01-17 00:00:00+00:00    49
2019-01-18 00:00:00+00:00    48
2019-01-20 00:00:00+00:00    36
2019-01-22 00:00:00+00:00     9
2019-01-23 00:00:00+00:00    25
2019-01-25 00:00:00+00:00    45
2019-01-27 00:00:00+00:00    41
2019-05-05 00:00:00+00:00    50
2019-05-06 00:00:00+00:00    27
2019-05-09 00:00:00+00:00    35
2019-05-10 00:00:00+00:00    23
2019-05-13 00:00:00+00:00    18
2019-05-17 00:00:00+00:00     8
Name: count, dtype: int64
mois
1    253
5    175
8     90
9    187
Name: count, dtype: int64
saison_article  Monsoon  Winter  Summer
t_out  mean        26.1    24.9    29.7
       min         24.3    23.1    29.1
       max         28.2    27.4    31.4
rh_out mean         NaN     NaN     NaN
       min          NaN     NaN     NaN
       max          Na

In [33]:
print(malik["clo"].agg(["mean", "min", "max"]).round(2))
print("Votes clo > 1.52 :", (malik["clo"] > 1.52).sum())

mean    0.56
min     0.15
max     2.34
Name: clo, dtype: float64
Votes clo > 1.52 : 13


# Texte 14 : 
Karyono (1996) - Jakarta, Indonésie et Tokyo, Japon - bureaux - AC+MM+NV - 596 voix 
https://doi.org/10.1080/00038628.1996.9696808

# Texte 15 : PDF ZOTERO
Akande et Adebamowo (2010) - Bauchi, Nigeria - logements - NV - 468 votes - palier max 1

In [46]:
masque = df["publication"].str.contains("Akande", na=False)
Akande = df[masque].copy()
print("Pays :", Akande["country"].value_counts())
print("Villes :", Akande["city"].value_counts())
print("climat :", Akande["climate"].value_counts())
print("saison :", Akande["season"].value_counts())
print("Type de Bâtiments :", Akande["building_type"].value_counts())
print("Bâtiments :", Akande["building_id"].unique())
print("Genre :", Akande["gender"].value_counts())
print("Types de ventilation :", Akande["cooling_type"].value_counts())

print("Nombre de votes :", len(Akande))

Pays : country
nigeria    468
Name: count, dtype: int64
Villes : city
bauchi    468
Name: count, dtype: int64
climat : climate
tropical wet savanna    468
Name: count, dtype: int64
saison : season
winter    320
spring    148
Name: count, dtype: int64
Type de Bâtiments : building_type
multifamily housing    468
Name: count, dtype: int64
Bâtiments : [558 559]
Genre : gender
male      357
female    111
Name: count, dtype: int64
Types de ventilation : cooling_type
naturally ventilated    468
Name: count, dtype: int64
Nombre de votes : 468


In [47]:
variables = list(meas.columns)

remplissage = Akande[variables].notna().mean() * 100
print(remplissage.round(1).sort_values().to_string())

clo                             0.0
ppd_ce                          0.0
activity_10                     0.0
activity_20                     0.0
activity_30                     0.0
activity_60                     0.0
thermal_acceptability           0.0
air_movement_acceptability      0.0
blind_curtain                   0.0
fan                             0.0
met                             0.0
window                          0.0
heater                          0.0
rh_out                          0.0
t_out_monthly                   0.0
t_out_isd                       0.0
rh_out_isd                      0.0
set                             0.0
pmv                             0.0
ppd                             0.0
pmv_ce                          0.0
door                            0.0
vel_l                           0.0
t_mot_isd                       0.0
vel_h                           0.0
timestamp                       0.0
subject_id                      0.0
ta_h                        

In [48]:
saisons_Akande = {"winter": "Dry Season", "spring": "Rainy Season"}
Akande["saison_article"] = Akande["season"].map(saisons_Akande)

print(Akande["saison_article"].value_counts())


stats = Akande.groupby("saison_article")[["ta", "t_out", "rh", "rh_out", "vel"]].agg(["mean", "max", "min"])
print(stats.round(1).T)

saison_article
Dry Season      320
Rainy Season    148
Name: count, dtype: int64
saison_article  Dry Season  Rainy Season
ta     mean           34.8          33.0
       max            39.0          39.0
       min            29.0          27.0
t_out  mean           38.0          37.6
       max            43.0          42.0
       min            30.0          30.0
rh     mean           69.2          70.9
       max            93.0          93.0
       min            18.5          34.0
rh_out mean            NaN           NaN
       max             NaN           NaN
       min             NaN           NaN
vel    mean            0.1           0.1
       max             3.1           3.1
       min             0.0           0.0


# Texte 17 : 
Sekhar et coll. (2003) - Singapour - bureaux - AC - 216 voix - palier max 3 
https://doi.org/10.1111/j.1600-0668.2003.00191.x

In [68]:
masque = df["publication"].str.contains("j.1600-0668.2003.00191", na=False)
Sekhar = df[masque].copy()
Sekhar["date"] = pd.to_datetime(Sekhar["timestamp"])
print(Sekhar["date"].value_counts().sort_index())
print("Pays :", Sekhar["country"].value_counts())
print("Villes :", Sekhar["city"].value_counts())
print("climat :", Sekhar["climate"].value_counts())
print("saison :", Sekhar["season"].value_counts())
print("Type de Bâtiments :", Sekhar["building_type"].value_counts())
print("Bâtiments :", Sekhar["building_id"].unique())
print("Genre :", Sekhar["gender"].value_counts())
print("Types de ventilation :", Sekhar["cooling_type"].value_counts())
print("____________________________________")
print("Nombre de votes :", len(Sekhar))

print("____________________________________")

variables = list(meas.columns)
remplissage = Sekhar[variables].notna().mean() * 100
print(remplissage.round(1).sort_values().to_string())

print("____________________________________")

print(Sekhar["season"].value_counts())
stats = Sekhar.groupby("season")[["ta", "tr", "rh", "vel"]].agg(["mean", "max", "min"])
print(stats.round(2).T)

Series([], Name: count, dtype: int64)
Pays : country
singapore    216
Name: count, dtype: int64
Villes : city
singapore    216
Name: count, dtype: int64
climat : climate
tropical rainforest    216
Name: count, dtype: int64
saison : season
summer    216
Name: count, dtype: int64
Type de Bâtiments : building_type
office    216
Name: count, dtype: int64
Bâtiments : [714]
Genre : Series([], Name: count, dtype: int64)
Types de ventilation : cooling_type
air conditioned    216
Name: count, dtype: int64
____________________________________
Nombre de votes : 216
____________________________________
clo                             0.0
activity_10                     0.0
activity_20                     0.0
activity_30                     0.0
activity_60                     0.0
thermal_acceptability           0.0
thermal_preference              0.0
air_movement_acceptability      0.0
air_movement_preference         0.0
blind_curtain                   0.0
fan                             0.0
ppd_ce

# Texte 18 : 
Schweiker et coll. (2020) - Kota Kinabalu, Malaisie - salle de classe - AC - 171 votes - palier max 1 
https://doi.org/10.1016/j.enbuild.2020.109761

In [73]:
masque = df["publication"].str.contains("j.enbuild.2020.109761", na=False)
Schweiker = df[masque].copy()
print("Dates :", Schweiker["timestamp"].value_counts().sort_index())
print("Pays :", Schweiker["country"].value_counts())
print("Villes :", Schweiker["city"].value_counts())
print("climat :", Schweiker["climate"].value_counts())
print("saison :", Schweiker["season"].value_counts())
print("Type de Bâtiments :", Schweiker["building_type"].value_counts())
print("Bâtiments :", Schweiker["building_id"].unique())
print("Genre :", Schweiker["gender"].value_counts())
print("Types de ventilation :", Schweiker["cooling_type"].value_counts())

print("____________________________________")

print("Nombre de votes :", len(Schweiker))

print("____________________________________")

variables = list(meas.columns)
remplissage = Schweiker[variables].notna().mean() * 100
print(remplissage.round(1).sort_values().to_string())

print("____________________________________")

print(Schweiker["season"].value_counts())
stats = Schweiker.groupby("season")[["ta", "tr", "rh", "vel"]].agg(["mean", "max", "min"])
print(stats.round(2).T)

Dates : timestamp
2017-12-15T00:00:00Z    46
2018-05-17T00:00:00Z    36
2018-05-18T00:00:00Z    37
2018-11-01T00:00:00Z    52
Name: count, dtype: int64
Pays : country
malaysia    171
Name: count, dtype: int64
Villes : city
kota kinabalu    171
Name: count, dtype: int64
climat : climate
tropical rainforest    171
Name: count, dtype: int64
saison : season
cool/dry    98
hot/wet     73
Name: count, dtype: int64
Type de Bâtiments : building_type
classroom    171
Name: count, dtype: int64
Bâtiments : [543 544 545 546]
Genre : gender
female       88
male         82
undefined     1
Name: count, dtype: int64
Types de ventilation : cooling_type
air conditioned    171
Name: count, dtype: int64
____________________________________
Nombre de votes : 171
____________________________________
clo                             0.0
thermal_comfort                 0.0
t_out                           0.0
rh_out                          0.0
ppd_ce                          0.0
met                            

# Texte 20 
Xavier (2000, thèse) - Brésil - bureaux - AC - 51 voix - palier max 4

In [80]:
masque = trop["publication"].str.contains("Xavier", na=False)
xavier = trop[masque].copy()
print("Dates :", xavier["timestamp"].value_counts().sort_index())
print("Pays :", xavier["country"].value_counts())
print("Villes :", xavier["city"].value_counts())
print("climat :", xavier["climate"].value_counts())
print("saison :", xavier["season"].value_counts())
print("Type de Bâtiments :", xavier["building_type"].value_counts())
print("Bâtiments :", xavier["building_id"].unique())
print("Genre :", xavier["gender"].value_counts())
print("Types de ventilation :", xavier["cooling_type"].value_counts())

print("____________________________________")

print("Nombre de votes :", len(xavier))

print("____________________________________")

variables = list(meas.columns)
remplissage = xavier[variables].notna().mean() * 100
print(remplissage.round(1).sort_values().to_string())

print("____________________________________")

print(xavier["season"].value_counts())
stats = xavier.groupby("season")[["ta", "tr", "rh", "vel"]].agg(["mean", "max", "min"])
print(stats.round(2).T)

Dates : Series([], Name: count, dtype: int64)
Pays : country
brazil    51
Name: count, dtype: int64
Villes : city
recife      27
brasilia    24
Name: count, dtype: int64
climat : climate
tropical monsoon        27
tropical wet savanna    24
Name: count, dtype: int64
saison : season
spring    27
autumn    24
Name: count, dtype: int64
Type de Bâtiments : building_type
office    51
Name: count, dtype: int64
Bâtiments : [789 797]
Genre : Series([], Name: count, dtype: int64)
Types de ventilation : cooling_type
air conditioned    51
Name: count, dtype: int64
____________________________________
Nombre de votes : 51
____________________________________
t_mot_isd                       0.0
vel_h                           0.0
vel_m                           0.0
vel_l                           0.0
rh_out                          0.0
t_out                           0.0
activity_10                     0.0
activity_20                     0.0
activity_30                     0.0
activity_60          

In [81]:
masque = df["publication"].str.contains("Xavier", na=False)
xavier = df[masque].copy()
print("Dates :", xavier["timestamp"].value_counts().sort_index())
print("Pays :", xavier["country"].value_counts())
print("Villes :", xavier["city"].value_counts())
print("climat :", xavier["climate"].value_counts())
print("saison :", xavier["season"].value_counts())
print("Type de Bâtiments :", xavier["building_type"].value_counts())
print("Bâtiments :", xavier["building_id"].unique())
print("Genre :", xavier["gender"].value_counts())
print("Types de ventilation :", xavier["cooling_type"].value_counts())

print("____________________________________")

print("Nombre de votes :", len(xavier))

print("____________________________________")

variables = list(meas.columns)
remplissage = xavier[variables].notna().mean() * 100
print(remplissage.round(1).sort_values().to_string())

print("____________________________________")

print(xavier["season"].value_counts())
stats = xavier.groupby("season")[["ta", "tr", "rh", "vel"]].agg(["mean", "max", "min"])
print(stats.round(2).T)

Dates : Series([], Name: count, dtype: int64)
Pays : country
brazil    279
Name: count, dtype: int64
Villes : city
florianopolis    228
recife            27
brasilia          24
Name: count, dtype: int64
climat : climate
humid subtropical       228
tropical monsoon         27
tropical wet savanna     24
Name: count, dtype: int64
saison : season
winter    117
spring     72
autumn     66
summer     24
Name: count, dtype: int64
Type de Bâtiments : building_type
office       209
classroom     70
Name: count, dtype: int64
Bâtiments : [789 790 791 792 793 794 795 796 797]
Genre : Series([], Name: count, dtype: int64)
Types de ventilation : cooling_type
air conditioned         209
naturally ventilated     70
Name: count, dtype: int64
____________________________________
Nombre de votes : 279
____________________________________
t_mot_isd                       0.0
vel_h                           0.0
vel_m                           0.0
vel_l                           0.0
rh_out                 

Af (Tropical Rainforest / Tropical Wet)
Am (Tropical Monsoon)
Aw or As (Tropical Savannah / Tropical Wet and Dry)

In [63]:
meta[meta["building_id"] == 714]

,building_id,building_id_inf,contributor,publication,region,country,city,lat,lon,climate,...,records,has_age,has_ec,has_timestamp,timezone,met_source,isd_station,isd_distance,database,quality_assurance
713,714,yes,Tham Kwok Wai,https://doi.org/10.1111/j.1600-0668.2003.00191.x,asia,singapore,singapore,1.352083,103.819836,tropical rainforest,...,216,no,no,no,Asia/Singapore,NaN,NaN,NaN,2.0,pass
